# Task 15: Enterprise LLM Gateway with Dynamic Rate-Limiting and Fallback Governance

## Objective

To design a robust LLM gateway that manages API rate limiting, model fallback, latency tracking, request handling, and performance metrics.

## Technologies / Tools Used

- Python 3.10+
- FastAPI
- Redis
- Docker
- Prometheus
- Grafana
- Google Colab

## Formula

### Token-Bucket Rate Limiting

\[
Tokens = \min(C,\ Tokens + R \times \Delta t)
\]

Where `C` is the bucket capacity and `R` is the refill rate.

### Model Fallback

\[
Primary\ Model \rightarrow 5xx\ Error \rightarrow Secondary\ Model
\]

## Step 1: Install Required Libraries

Install the libraries required for the LLM gateway.

In [1]:
!pip -q install fastapi uvicorn redis prometheus-client requests
!apt-get -qq update
!apt-get -qq install redis-server

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 6.7 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libjemalloc2:amd64.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../0-libjemalloc2_5.2.1-4ubuntu1_amd64.deb ...
Unpacking libjemalloc2:amd64 (5.2.1-4ubuntu1) ...
Selecting previously unselected package liblua5.1-0:amd64.
Preparing to unpack .../1-liblua5.1-0_5.1.5-8.1build4_amd64.deb ...
Unpacking liblua5.1-0:amd64 (5.1.5-8.1build4) ...
Selecting previously unselected package liblzf1:amd64.
Preparing to unpack .../2-liblzf1_3.6-3_amd64.deb ...
Unpacking liblzf1:amd64 (3.6-3) ...
Selecting previously unselected package lua-bitop:amd64.
Preparing to unpack .../3-lua-bitop_1.0.2-5_amd64.deb ...
Unpacking lua-bitop:amd64 (1.0.2-5) ...
Sel

## Step 2: Start Redis

Start Redis and verify that the Redis server is running.

In [2]:
!redis-server --daemonize yes

import redis

r = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)

print("Redis connected:", r.ping())

Redis connected: True


## Step 3: Create the LLM Gateway

Create a FastAPI gateway with Redis-based rate limiting, model fallback, and Prometheus metrics.

In [4]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel
from prometheus_client import Counter, Histogram, generate_latest
from fastapi.responses import Response
import time
import random

app = FastAPI(title="Enterprise LLM Gateway")

REQUESTS = Counter(
    "llm_gateway_requests_total",
    "Total LLM requests"
)

LATENCY = Histogram(
    "llm_gateway_latency_seconds",
    "LLM gateway latency"
)

class RequestData(BaseModel):
    prompt: str
    user_id: str = "user1"

def rate_limit(user_id, limit=5):
    key = f"rate:{user_id}"
    count = r.incr(key)

    if count == 1:
        r.expire(key, 60)

    return count <= limit

def primary_model(prompt):
    if random.random() < 0.3:
        raise Exception("Primary model unavailable")
    return f"Primary model response: {prompt}"

def secondary_model(prompt):
    return f"Secondary model response: {prompt}"

@app.post("/generate")
def generate(data: RequestData):

    start = time.time()
    REQUESTS.inc()

    if not rate_limit(data.user_id):
        raise HTTPException(
            status_code=429,
            detail="Rate limit exceeded"
        )

    try:
        response = primary_model(data.prompt)
        model = "Primary Model"

    except Exception:
        response = secondary_model(data.prompt)
        model = "Secondary Model (Fallback)"

    latency = time.time() - start
    LATENCY.observe(latency)

    return {
        "response": response,
        "model": model,
        "latency": round(latency, 4)
    }

@app.get("/metrics")
def metrics():
    return Response(
        generate_latest(),
        media_type="text/plain"
    )

print("LLM Gateway created successfully.")

LLM Gateway created successfully.


## Step 4: Test Rate Limiting and Model Fallback

Send multiple requests to verify rate limiting and automatic fallback.

In [5]:
client = TestClient(app)

for i in range(7):

    response = client.post(
        "/generate",
        json={
            "prompt": f"Explain Generative AI {i}",
            "user_id": "student1"
        }
    )

    print(
        "Request:", i + 1,
        "| Status:", response.status_code,
        "|", response.json()
    )

Request: 1 | Status: 200 | {'response': 'Secondary model response: Explain Generative AI 0', 'model': 'Secondary Model (Fallback)', 'latency': 0.0008}
Request: 2 | Status: 200 | {'response': 'Primary model response: Explain Generative AI 1', 'model': 'Primary Model', 'latency': 0.0004}
Request: 3 | Status: 200 | {'response': 'Secondary model response: Explain Generative AI 2', 'model': 'Secondary Model (Fallback)', 'latency': 0.0003}
Request: 4 | Status: 200 | {'response': 'Primary model response: Explain Generative AI 3', 'model': 'Primary Model', 'latency': 0.0003}
Request: 5 | Status: 200 | {'response': 'Primary model response: Explain Generative AI 4', 'model': 'Primary Model', 'latency': 0.0003}
Request: 6 | Status: 429 | {'detail': 'Rate limit exceeded'}
Request: 7 | Status: 429 | {'detail': 'Rate limit exceeded'}


## Step 5: Check Prometheus Metrics

Display the performance metrics collected by the gateway.

In [6]:
response = client.get("/metrics")

print(response.text[:2500])

# HELP python_gc_objects_collected_total Objects collected during gc
# TYPE python_gc_objects_collected_total counter
python_gc_objects_collected_total{generation="0"} 2872.0
python_gc_objects_collected_total{generation="1"} 374.0
python_gc_objects_collected_total{generation="2"} 196.0
# HELP python_gc_objects_uncollectable_total Uncollectable objects found during GC
# TYPE python_gc_objects_uncollectable_total counter
python_gc_objects_uncollectable_total{generation="0"} 0.0
python_gc_objects_uncollectable_total{generation="1"} 0.0
python_gc_objects_uncollectable_total{generation="2"} 0.0
# HELP python_gc_collections_total Number of times this generation was collected
# TYPE python_gc_collections_total counter
python_gc_collections_total{generation="0"} 368.0
python_gc_collections_total{generation="1"} 33.0
python_gc_collections_total{generation="2"} 3.0
# HELP python_info Python platform information
# TYPE python_info gauge
python_info{implementation="CPython",major="3",minor="12",pa

## Step 6: Create Docker Deployment Files

Create the Docker configuration required to containerize the gateway.

In [7]:
%%writefile Dockerfile

FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile


In [8]:
%%writefile docker-compose.yml

services:

  gateway:
    build: .
    ports:
      - "8000:8000"
    depends_on:
      - redis

  redis:
    image: redis:7

  prometheus:
    image: prom/prometheus
    ports:
      - "9090:9090"

  grafana:
    image: grafana/grafana
    ports:
      - "3000:3000"

Writing docker-compose.yml


## Step 7: Display Gateway Architecture

The following architecture shows the flow between the client, gateway, Redis, LLM models, and monitoring systems.

In [9]:
print("""
                Client
                  |
                  v
          FastAPI LLM Gateway
                  |
          +-------+-------+
          |               |
        Redis          LLM Router
    Rate Limiting          |
                            v
                     Primary Model
                            |
                         5xx Error
                            |
                            v
                    Secondary Model
                            |
                            v
                  Prometheus Metrics
                            |
                            v
                         Grafana
""")


                Client
                  |
                  v
          FastAPI LLM Gateway
                  |
          +-------+-------+
          |               |
        Redis          LLM Router
    Rate Limiting          |
                            v
                     Primary Model
                            |
                         5xx Error
                            |
                            v
                    Secondary Model
                            |
                            v
                  Prometheus Metrics
                            |
                            v
                         Grafana



## Conclusion

The enterprise LLM gateway was successfully implemented with FastAPI and Redis. The system provides dynamic rate limiting, automatic model fallback, latency tracking, Prometheus metrics, and Docker-based deployment configuration.